In [142]:
import torch
from torch import nn
import numpy as np
import json

In [143]:
class BengioLM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, context_size = 4):
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.context_size = context_size
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.W = nn.Linear(context_size * embedding_dim, hidden_dim)
        self.tanh = nn.Tanh()
        self.U = nn.Linear(hidden_dim, vocab_size)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, input_token_ids):
        e = self.embedding(input_token_ids)
        e = e.view(e.size(0),-1)
        h = self.W(e)
        h = self.tanh(h)
        logits = self.U(h)
        probabilities = self.softmax(logits)
        log_probabilities = nn.LogSoftmax(dim=-1)(logits)
        return log_probabilities


In [144]:
class Vocabulary:
    def __init__(self):
        self._token_by_id = {} #id => token
        self._id_by_token = {} #token => id

    def add_token(self, token: str):
        if token in self._id_by_token:
            return
        next_id = len(self._token_by_id)
        self._token_by_id[next_id] = token
        self._id_by_token[token] = next_id
    @classmethod
    def from_file(self, path):
        vocab = Vocabulary()
        with open(path, encoding="utf-8") as file:
            for line in file:
                tokens = line.split()
                for token in tokens:
                    vocab.add_token(token)
        return vocab


    def token_by_id(self, token_id):
        return self._token_by_id[token_id]
    def id_by_token(self, token):
        return self._id_by_token[token]
    def __len__(self):
        return len(self._token_by_id)

    def save(self, path):
        with open(path, "w", encoding="utf-8") as file:
            file.write(json.dumps(self._id_by_token, ensure_ascii=False))

    @staticmethod
    def load(path):
        vocab = Vocabulary()
        with open(path, encoding="utf-8") as file:
            vocab._id_by_token = json.loads(file.read())
            vocab._token_by_id = {id: token for token, id in vocab._id_by_token.items()}
        return vocab


In [145]:
from typing import List, Tuple

def tokenize(line: str, vocab) -> List[int]:
    """Повертає індекси токенів з тексту"""
    tokens = line.split()
    indexes = [vocab.id_by_token(token) for token in tokens]
    return indexes


def prepare_train_examples(dataset, vocab, context_size) -> List[Tuple[List[int], int]]:
    result = []
    for line in dataset:
        token_ids = tokenize(line, vocab)

        for i in range(len(token_ids) - context_size - 1):
            context = token_ids[i: i + context_size]
            target = token_ids[i + context_size]
            result.append((context, target))

    return result


In [146]:
vocab = Vocabulary.from_file("./train.bpe.txt")
print(vocab._token_by_id)
print(vocab.id_by_token("спить"))
print(vocab.token_by_id(1))
dataset = open("./train.bpe.txt", encoding="utf-8").readlines()
train_examples = prepare_train_examples(dataset, vocab, context_size=2)
print(train_examples)

{0: 'Кіт', 1: 'спить', 2: 'на', 3: 'кухн@@', 4: 'і', 5: 'Собака', 6: 'біг@@', 7: 'ав', 8: 'у', 9: 'саду', 10: 'Птах', 11: 'лет@@', 12: 'ить', 13: 'над', 14: 'б@@', 15: 'удин@@', 16: 'ком', 17: 'Кролик', 18: 'хова@@', 19: 'в@@', 20: 'ся', 21: 'під', 22: 'сто@@', 23: 'л@@', 24: 'ом', 25: 'Кінь', 26: 'іде', 27: 'п@@', 28: 'о', 29: 'пол@@', 30: 'ю', 31: 'Корова', 32: 'я@@', 33: 'ла', 34: 'біля', 35: 'са@@', 36: 'ра@@', 37: 'сидить', 38: 'ст@@', 39: 'і@@', 40: 'ь@@', 41: 'ц@@', 42: 'сп@@', 43: 'дв@@', 44: 'ере@@', 45: 'й', 46: 'К@@', 47: 'а@@', 48: 'чка', 49: 'ва@@', 50: 'є', 51: 'ста@@', 52: 'ку', 53: 'Л@@', 54: 'исиця', 55: 'через', 56: 'лі@@', 57: 'с', 58: 'М@@', 59: 'иша', 60: 'ховається', 61: 'за', 62: 'ко@@', 63: 'ро@@', 64: 'кою', 65: 'В@@', 66: 'івця', 67: 'ходила', 68: 'д@@', 69: 'оро@@', 70: 'г@@', 71: 'урка', 72: 'ї@@', 73: 'с@@', 74: 'ть', 75: 'дворі', 76: 'навколо', 77: 'по@@', 78: 'ля', 79: 'ч@@', 80: 'е@@', 81: 'к@@', 82: 'в', 83: 'р@@', 84: 'и@@', 85: 'орі', 86: 'ж@@', 87: '

In [147]:
CONTEXT_SIZE = 2
def train():
    vocab = Vocabulary.from_file("./train.bpe.txt")
    model = BengioLM(vocab_size = len(vocab), embedding_dim=8, hidden_dim=6, context_size = CONTEXT_SIZE)

    optimizer = torch.optim.SGD(model.parameters(), 0.1)

    for epoch in range(50):
        epoch_loss = 0.0
        for context, target in train_examples:
            context = torch.tensor(context).unsqueeze(0)
            log_probs = model(context)
            loss = -log_probs.squeeze(0)[target] ##negative log likelihood loss

            #Виконання кроку Back-Propagation
            epoch_loss += loss.item()
            loss.backward()

            optimizer.step()
            optimizer.zero_grad()
        average_loss = epoch_loss / len(train_examples)
        print(f"Epoch {epoch + 1}, Average Loss: {average_loss:.4f}")
    return model



In [148]:
model = train()
torch.save(model, "./model.pt")

Epoch 1, Average Loss: 4.6260
Epoch 2, Average Loss: 4.2172
Epoch 3, Average Loss: 3.8509
Epoch 4, Average Loss: 3.5071
Epoch 5, Average Loss: 3.1973
Epoch 6, Average Loss: 2.9199
Epoch 7, Average Loss: 2.6841
Epoch 8, Average Loss: 2.4775
Epoch 9, Average Loss: 2.2936
Epoch 10, Average Loss: 2.1360
Epoch 11, Average Loss: 2.0033
Epoch 12, Average Loss: 1.8965
Epoch 13, Average Loss: 1.8003
Epoch 14, Average Loss: 1.7190
Epoch 15, Average Loss: 1.6433
Epoch 16, Average Loss: 1.5754
Epoch 17, Average Loss: 1.5168
Epoch 18, Average Loss: 1.4586
Epoch 19, Average Loss: 1.4009
Epoch 20, Average Loss: 1.3579
Epoch 21, Average Loss: 1.3221
Epoch 22, Average Loss: 1.2914
Epoch 23, Average Loss: 1.2440
Epoch 24, Average Loss: 1.2308
Epoch 25, Average Loss: 1.1950
Epoch 26, Average Loss: 1.1690
Epoch 27, Average Loss: 1.1636
Epoch 28, Average Loss: 1.1626
Epoch 29, Average Loss: 1.1736
Epoch 30, Average Loss: 1.1892
Epoch 31, Average Loss: 1.2341
Epoch 32, Average Loss: 1.1640
Epoch 33, Average

In [149]:
model = torch.load("./model.pt", weights_only=False)

In [150]:
import math
def text_log_probability(text, model, vocab, context_size):
    examples = prepare_train_examples([text], vocab, context_size)
    text_prob = []
    for context, target in examples:
        context = torch.tensor(context).unsqueeze(0)
        log_prob = model(context)
        log_prob = log_prob.squeeze(0)[target]
        text_prob.append(log_prob.item())
    log_prob_avg = sum(text_prob) / len(text_prob)
    return log_prob_avg

def perplexity(text, model, vocab, context_size):
    return math.exp(-text_log_probability(text, model, vocab, context_size))

In [151]:
vocab.save("vocab.txt")

In [152]:
vocab = Vocabulary.load("./vocab.txt")
print(vocab._token_by_id)

{0: 'Кіт', 1: 'спить', 2: 'на', 3: 'кухн@@', 4: 'і', 5: 'Собака', 6: 'біг@@', 7: 'ав', 8: 'у', 9: 'саду', 10: 'Птах', 11: 'лет@@', 12: 'ить', 13: 'над', 14: 'б@@', 15: 'удин@@', 16: 'ком', 17: 'Кролик', 18: 'хова@@', 19: 'в@@', 20: 'ся', 21: 'під', 22: 'сто@@', 23: 'л@@', 24: 'ом', 25: 'Кінь', 26: 'іде', 27: 'п@@', 28: 'о', 29: 'пол@@', 30: 'ю', 31: 'Корова', 32: 'я@@', 33: 'ла', 34: 'біля', 35: 'са@@', 36: 'ра@@', 37: 'сидить', 38: 'ст@@', 39: 'і@@', 40: 'ь@@', 41: 'ц@@', 42: 'сп@@', 43: 'дв@@', 44: 'ере@@', 45: 'й', 46: 'К@@', 47: 'а@@', 48: 'чка', 49: 'ва@@', 50: 'є', 51: 'ста@@', 52: 'ку', 53: 'Л@@', 54: 'исиця', 55: 'через', 56: 'лі@@', 57: 'с', 58: 'М@@', 59: 'иша', 60: 'ховається', 61: 'за', 62: 'ко@@', 63: 'ро@@', 64: 'кою', 65: 'В@@', 66: 'івця', 67: 'ходила', 68: 'д@@', 69: 'оро@@', 70: 'г@@', 71: 'урка', 72: 'ї@@', 73: 'с@@', 74: 'ть', 75: 'дворі', 76: 'навколо', 77: 'по@@', 78: 'ля', 79: 'ч@@', 80: 'е@@', 81: 'к@@', 82: 'в', 83: 'р@@', 84: 'и@@', 85: 'орі', 86: 'ж@@', 87: '

In [153]:
text = "К@@ а@@ чка п@@ л@@ а@@ ва@@ є у ста@@ в@@ ку"
prob = text_log_probability(text, model, vocab, context_size=2)
print(np.exp(prob))

0.5000635668886001


In [154]:
validation_texts = [
    # Точні приклади з тренувальної вибірки
    "Кіт спить на кухн@@ і",
    "Собака біг@@ ав у саду",
    "Кролик хова@@ в@@ ся під сто@@ л@@ ом",

    # Нові речення, схожі на тренувальні
    "Птах спить на кухн@@ і",
    "Кіт біг@@ ав у саду",
    "Кінь хова@@ в@@ ся під сто@@ л@@ ом",

    # Менш схожі та менш природні послідовності
    "Собака лет@@ ить над б@@ удин@@ ком",
    "кухн@@ і спить Кролик над саду",
    "Птах під сто@@ л@@ ом біг@@ ав у"
]

for text in validation_texts:
    log_probability = text_log_probability(text, model, vocab, context_size=2)
    probability = np.exp(log_probability)
    print(f"{text} -> {probability:.6f}")


Кіт спить на кухн@@ і -> 0.388888
Собака біг@@ ав у саду -> 0.535134
Кролик хова@@ в@@ ся під сто@@ л@@ ом -> 0.496217
Птах спить на кухн@@ і -> 0.075310
Кіт біг@@ ав у саду -> 0.129131
Кінь хова@@ в@@ ся під сто@@ л@@ ом -> 0.139504
Собака лет@@ ить над б@@ удин@@ ком -> 0.053222
кухн@@ і спить Кролик над саду -> 0.000028
Птах під сто@@ л@@ ом біг@@ ав у -> 0.003258
